# 📊 Results Visualization & Pipeline Analysis

This notebook provides a comprehensive analysis of the results obtained from various model architectures and quantum circuit configurations for the Video Question Generation task.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load data
csv_path = '../runs.csv'
if not os.path.exists(csv_path):
    # Try relative to root if notebook is run from root
    csv_path = 'runs.csv'

df = pd.read_csv(csv_path)
df = df.dropna(subset=['Model'])
print("Loaded Results Data:")
display(df)

## 🏗️ Pipeline Architecture

The final pipeline consists of multimodal feature extraction, quantum-enhanced fusion, and a generative head with alignment loss.

```mermaid
graph TD
    subgraph Data_Source
        V[Video Segment] --> C[CLIP Embeddings]
        V --> E[EfficientNet Embeddings]
        T[Textual Metadata] --> TXT[BART Encoder]
    end

    subgraph Quantum_Fusion
        C & E --> CA[Cross-Attention Mechanism]
        CA --> VQC[Variational Quantum Circuit - RTheta]
        VQC --> VC[Quantum Video Context]
    end

    subgraph Generator
        TXT & VC --> MCA[Multimodal Cross-Attention]
        MCA --> BD[BART Decoder]
        BD --> Q[Generated Question]
    end

    subgraph Verification
        Q --> AL[Alignment Loss]
        T5[Reference T5 Model] --> AL
    end
```

## 📈 Performance Analysis

In [ ]:
# Filter Q-BART results for scaling analysis
qbart_df = df[df['Model'] == 'QBart'].copy()
qbart_df['Size'] = qbart_df['Size'].astype(int)
qbart_df = qbart_df.sort_values('Size')

# Plot 1: ROUGE-L Scaling with Qubits
plt.figure(figsize=(10, 5))
sns.lineplot(data=qbart_df, x='Size', y='rougeL', marker='o', label='ROUGE-L')
sns.lineplot(data=qbart_df, x='Size', y='bleu1', marker='s', label='BLEU-1')
plt.title('Performance Scaling with Quantum Circuit Size (Qubits)')
plt.xlabel('Number of Qubits')
plt.ylabel('Score')
plt.xticks([8, 16, 32])
plt.legend()
plt.show()

# Plot 2: Model Comparison (ROUGE-L)
plt.figure(figsize=(12, 6))
comparison_df = df[df['Size'].isna() | (df['Size'] == 32)].copy()
comparison_df['Model_Label'] = comparison_df.apply(lambda x: f"{x['Model']} (Q32)" if x['Size'] == 32 else x['Model'], axis=1)

sns.barplot(data=comparison_df, x='Model_Label', y='rougeL', hue='Model_Label', palette='viridis', legend=False)
plt.title('Comparison of ROUGE-L across different Architectures')
plt.ylabel('ROUGE-L Score')
plt.show()

# Plot 3: Semantic Alignment (BERTScore)
plt.figure(figsize=(12, 6))
sns.barplot(data=comparison_df, x='Model_Label', y='bert', hue='Model_Label', palette='magma', legend=False)
plt.title('Semantic Alignment (BERTScore) Comparison')
plt.ylabel('BERTScore (F1)')
plt.ylim(0.5, 1.0) # Zoom in for semantic differences
plt.show()

## 💡 Key Observations
1. **Quantum Scaling**: A clear positive correlation exists between the number of qubits in the fusion layer and the generation quality (ROUGE-L and BLEU).
2. **Semantic Superiority**: The Quantum-enhanced BART model (32 qubits) achieves the highest BERTScore, indicating better contextual alignment with the video content compared to larger non-quantum baselines.
3. **Efficiency**: Q-BART 32 achieves performance near large transformer baselines while utilizing significantly smaller classical parameter counts in the fusion block.